In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from glob import glob
from tqdm import tqdm

In [ ]:
MASK_DIR = (
    "../outputs/vit/"
    "attention_rollout/masks"
)

mask_files = glob(
    os.path.join(
        MASK_DIR,
        "*.npy"
    )
)

print(
    f"Total masks found: "
    f"{len(mask_files)}"
)

In [ ]:
def compute_entropy(mask):

    saliency = mask.flatten()

    saliency = saliency - saliency.min()

    saliency = saliency / (
        saliency.sum() + 1e-8
    )

    saliency = saliency[
        saliency > 0
    ]

    entropy = -np.sum(

        saliency *
        np.log(saliency + 1e-8)

    )

    return entropy

In [ ]:
results = []

for path in tqdm(mask_files):

    mask = np.load(path)

    entropy = compute_entropy(mask)

    filename = os.path.basename(path)

    true_cls = filename.split("_")[2]

    pred_cls = filename.split("_")[4]

    category = (
        "Correct"
        if true_cls == pred_cls
        else "Misclassified"
    )

    results.append({

        "file": filename,

        "entropy": entropy,

        "true": true_cls,

        "pred": pred_cls,

        "category": category
    })

print("Entropy computation complete.")

In [ ]:
df = pd.DataFrame(results)

df.head()

In [ ]:
print("\n===== ENTROPY SUMMARY =====\n")

print(

    df.groupby(
        "category"
    )["entropy"].describe()

)

In [ ]:
plt.figure(figsize=(8, 6))

df.boxplot(

    column="entropy",

    by="category"
)

plt.title(
    "Attention Rollout Entropy"
)

plt.suptitle("")

plt.ylabel("Entropy")

plt.show()

In [ ]:
plt.figure(figsize=(8, 6))

for category in df[
    "category"
].unique():

    subset = df[
        df["category"] == category
    ]

    plt.hist(

        subset["entropy"],

        bins=10,

        alpha=0.6,

        label=category
    )

plt.xlabel("Entropy")

plt.ylabel("Frequency")

plt.title(
    "Entropy Distribution"
)

plt.legend()

plt.show()

In [ ]:
SAVE_PATH = (
    "../outputs/vit/"
    "vit_entropy_results.csv"
)

df.to_csv(
    SAVE_PATH,
    index=False
)

print(
    f"Saved results to:\n"
    f"{SAVE_PATH}"
)